# Практическое занятие 2. Извлечение данных из заявок на аренду жилья

Решение: LangChain + GigaChat извлекают количество проживающих из
неструктурированных текстов заявок.

Ключ доступа хранится в файле `.env` в переменной `GIGA_KEY` и не попадает
в репозиторий.

## 1. Подключение к GigaChat

In [1]:
import os

from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat

load_dotenv()
giga_key = os.getenv("GIGA_KEY")

if not giga_key:
    raise ValueError("Ключ GIGA_KEY не найден в .env")

llm = GigaChat(
    credentials=giga_key,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000,
)

print(llm.invoke("Привет! Как дела?").content)

Привет! Всё отлично, спасибо. Готов обсудить что-нибудь интересное или помочь разобраться с задачей. Чем займёмся?


## 2. Базовый промпт и цепочка LangChain

Промпт формулирует задачу, модель её выполняет, парсер `StrOutputParser`
возвращает результат как строку.

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:""",
)

chain = basic_prompt | llm | StrOutputParser()

test_texts = [
    "Ищу квартиру для семьи из четырех человек",
    "Нужна студия для одного",
    "Семья с двумя детьми ищет жилье",
]

for text in test_texts:
    print(f"Текст: {text}")
    print(f"Результат: {chain.invoke({'text': text})}")
    print("---")

Текст: Ищу квартиру для семьи из четырех человек


Результат: 4
---
Текст: Нужна студия для одного


Результат: 1
---
Текст: Семья с двумя детьми ищет жилье


Результат: 4
---


## 3. Обработка моих 15 заявок (вариант 17)

Заявки берутся из файла `rental_17.csv`. Ответы модели собираются в столбец
`result`, итоговая таблица сохраняется в `rental_with_results.csv`.

In [3]:
import pandas as pd

df = pd.read_csv("rental_17.csv", sep=";")

results = []
for _, row in df.iterrows():
    text = row["text"]
    try:
        results.append(chain.invoke({"text": text}))
    except Exception as error:
        results.append(f"ERROR: {error}")

df["result"] = results
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")
df[["amount", "result"]]

,amount,result
0,6,6
1,2,2
2,4,4
3,3,3
4,3,3
5,3,3
6,3,3
7,4,4
8,4,4
9,2,2
